# 03 - State-Space Regression — UK Gas Consumption

## Introduction

A **regression model in state-space form** represents the classical linear model:

$$
y_t = X_t \beta_t + \varepsilon_t, \quad \varepsilon_t \sim N(0, \sigma^2_\varepsilon)
$$

In the **fixed-coefficient** case, $\beta_t = \beta$ for all $t$ (constant),
and the state-space model reduces to classical OLS.

In the **time-varying parameter** (TVP) case:
$$
\beta_t = \beta_{t-1} + \eta_t, \quad \eta_t \sim N(0, Q)
$$

The coefficients follow a random walk, allowing them to evolve smoothly over time.
When $Q = 0$ (no state disturbance), the TVP model recovers the fixed-coefficient OLS.

We apply both models to **UK quarterly gas consumption** (1960-1986) with
trend and seasonal regressors, demonstrating:
- SSM with fixed coefficients = OLS equivalence
- Time-varying coefficients to capture structural change
- Model comparison via AIC/BIC

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from kalmanbox.models.regression_ssm import RegressionSSM
from kalmanbox.models.tvp import TimeVaryingParameters
from kalmanbox.datasets import load_dataset

import statsmodels.api as sm

import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print('Imports OK')

In [ ]:
# Load UK gas consumption data and construct regressors
df = load_dataset('uk_gas')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'Period: {df["quarter"].iloc[0]} to {df["quarter"].iloc[-1]}')

y = df['gas'].to_numpy(dtype=np.float64)
n = len(y)

# Construct regressors: intercept + trend + 3 seasonal dummies (Q2, Q3, Q4)
intercept = np.ones(n)
trend = np.arange(1, n + 1, dtype=np.float64)

# Seasonal dummies from quarter labels
quarters = df['quarter'].str[-1].astype(int).to_numpy()
dummy_q2 = (quarters == 2).astype(np.float64)
dummy_q3 = (quarters == 3).astype(np.float64)
dummy_q4 = (quarters == 4).astype(np.float64)

X = np.column_stack([intercept, trend, dummy_q2, dummy_q3, dummy_q4])
regressor_names = ['intercept', 'trend', 'Q2', 'Q3', 'Q4']

print(f'\nNumber of observations: {n}')
print(f'Number of regressors: {X.shape[1]}')
print(f'Regressor names: {regressor_names}')
print(f'\nGas consumption statistics:')
print(f'  Mean: {y.mean():.1f}')
print(f'  Std:  {y.std():.1f}')
print(f'  Min:  {y.min():.1f}, Max: {y.max():.1f}')

# Quick visualization
fig, ax = plt.subplots(figsize=(14, 5))
# Create date index from quarter strings
date_idx = pd.PeriodIndex(df['quarter'], freq='Q').to_timestamp()
ax.plot(date_idx, y, 'k-', linewidth=0.8)
ax.set_xlabel('Date')
ax.set_ylabel('Gas Consumption')
ax.set_title('UK Quarterly Gas Consumption (1960-1986)')
plt.tight_layout()
plt.show()

In [ ]:
# Model 1: Regression SSM with fixed coefficients
# This should be equivalent to OLS
model_fixed = RegressionSSM(y, X)
results_fixed = model_fixed.fit()

print('Model 1: Fixed-Coefficient Regression SSM')
print('=' * 55)
print(results_fixed.summary())

print(f'\nEstimated coefficients:')
for name, val, se in zip(regressor_names, results_fixed.params[:-1], results_fixed.se[:-1]):
    t_stat = val / se if se > 0 else np.nan
    print(f'  {name:12s}: {val:10.4f} (SE={se:.4f}, t={t_stat:.2f})')
print(f'  {"sigma2":12s}: {results_fixed.params[-1]:10.4f}')
print(f'\nLog-likelihood: {results_fixed.loglike:.4f}')

In [ ]:
# Model 2: Time-Varying Parameters SSM
# beta_t = beta_{t-1} + eta_t (random walk coefficients)
model_tvp = TimeVaryingParameters(y, X, q_type='diagonal')
results_tvp = model_tvp.fit()

print('Model 2: Time-Varying Parameter Regression')
print('=' * 55)
print(f'Parameters: {results_tvp.param_names}')
for name, val in zip(results_tvp.param_names, results_tvp.params):
    print(f'  {name:20s} = {val:.6f}')
print(f'\nLog-likelihood: {results_tvp.loglike:.4f}')

In [ ]:
# Comparison: Fixed SSM vs OLS (should be equivalent)
# OLS reference using statsmodels
ols_model = sm.OLS(y, X)
ols_results = ols_model.fit()

print('Comparison: SSM Fixed Coefficients vs OLS')
print('=' * 70)

comparison_df = pd.DataFrame({
    'Regressor': regressor_names,
    'SSM (kalmanbox)': [f'{v:.4f}' for v in results_fixed.params[:-1]],
    'OLS (statsmodels)': [f'{v:.4f}' for v in ols_results.params],
    'Difference': [f'{abs(a-b):.6f}' for a, b in zip(results_fixed.params[:-1], ols_results.params)],
})
print(comparison_df.to_string(index=False))

# Sigma2 comparison
ssm_sigma2 = results_fixed.params[-1]
ols_sigma2_mle = np.sum(ols_results.resid**2) / n  # MLE estimate (not unbiased)
print(f'\nError variance:')
print(f'  SSM sigma2: {ssm_sigma2:.4f}')
print(f'  OLS sigma2 (MLE): {ols_sigma2_mle:.4f}')
print(f'  Difference: {abs(ssm_sigma2 - ols_sigma2_mle):.6f}')

# Log-likelihood comparison
print(f'\nLog-likelihood:')
print(f'  SSM: {results_fixed.loglike:.4f}')
print(f'  OLS: {ols_results.llf:.4f}')

# Verify equivalence
coef_diff = np.max(np.abs(results_fixed.params[:-1] - ols_results.params))
print(f'\nMax coefficient difference: {coef_diff:.8f}')
if coef_diff < 0.01:
    print('VERIFIED: SSM fixed coefficients match OLS estimates')
else:
    print(f'Note: Small differences due to estimation approach')

In [ ]:
# Time-varying coefficients: evolution over time
smoothed_states = results_tvp.smoothed_state  # (nobs, k_regressors)

fig, axes = plt.subplots(3, 2, figsize=(16, 12))

for i, (name, ax) in enumerate(zip(regressor_names, axes.flat)):
    tvp_coef = smoothed_states[:, i]
    tvp_se = np.sqrt(results_tvp.smoothed_cov[:, i, i])
    
    ax.plot(date_idx, tvp_coef, 'b-', linewidth=1.2, label='TVP (smoothed)')
    ax.fill_between(date_idx, tvp_coef - 1.96 * tvp_se, tvp_coef + 1.96 * tvp_se,
                    alpha=0.2, color='blue')
    ax.axhline(results_fixed.params[i], color='red', linestyle='--',
               linewidth=1, label=f'OLS = {results_fixed.params[i]:.2f}')
    ax.set_title(f'Coefficient: {name}')
    ax.set_ylabel('Value')
    ax.legend(fontsize=8)

# Hide extra subplot if odd number of regressors
if len(regressor_names) < len(axes.flat):
    axes.flat[-1].set_visible(False)

plt.suptitle('Time-Varying vs Fixed Coefficients', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Test: Is sigma2_eta significantly > 0?
# If all sigma2_beta_i are close to zero, the fixed model is sufficient
print('Test: Are time-varying coefficients significant?')
print('=' * 60)

sigma2_obs = results_tvp.params[0]
print(f'sigma2_obs = {sigma2_obs:.6f}')
print()

for i, name in enumerate(regressor_names):
    sigma2_beta_i = results_tvp.params[1 + i]
    ratio = sigma2_beta_i / sigma2_obs if sigma2_obs > 0 else np.inf
    # Signal-to-noise ratio for this coefficient
    print(f'  sigma2_{name:12s} = {sigma2_beta_i:.6f}  '
          f'(ratio to obs: {ratio:.4f})')

total_sigma2_beta = sum(results_tvp.params[1:1+len(regressor_names)])
print(f'\nTotal state innovation variance: {total_sigma2_beta:.6f}')
print(f'Observation variance: {sigma2_obs:.6f}')
print(f'Signal-to-noise ratio: {total_sigma2_beta/sigma2_obs:.4f}' if sigma2_obs > 0 else '')

if total_sigma2_beta > 0.01 * sigma2_obs:
    print('\nConclusion: Time-varying coefficients appear to be non-trivial.')
    print('The data suggests structural change in the relationship.')
else:
    print('\nConclusion: State innovation variances are near zero.')
    print('The fixed-coefficient model may be sufficient.')

In [ ]:
# Model comparison: AIC / BIC
n_params_fixed = len(results_fixed.params)
n_params_tvp = len(results_tvp.params)

# AIC = -2*LL + 2*k, BIC = -2*LL + k*log(n)
aic_fixed = -2 * results_fixed.loglike + 2 * n_params_fixed
bic_fixed = -2 * results_fixed.loglike + n_params_fixed * np.log(n)

aic_tvp = -2 * results_tvp.loglike + 2 * n_params_tvp
bic_tvp = -2 * results_tvp.loglike + n_params_tvp * np.log(n)

comparison_ic = pd.DataFrame({
    'Model': ['Fixed Coefficients (SSM)', 'Time-Varying (TVP)', 'OLS (statsmodels)'],
    'Log-Likelihood': [f'{results_fixed.loglike:.2f}', f'{results_tvp.loglike:.2f}', f'{ols_results.llf:.2f}'],
    'Num. Params': [n_params_fixed, n_params_tvp, len(ols_results.params) + 1],
    'AIC': [f'{aic_fixed:.2f}', f'{aic_tvp:.2f}', f'{ols_results.aic:.2f}'],
    'BIC': [f'{bic_fixed:.2f}', f'{bic_tvp:.2f}', f'{ols_results.bic:.2f}'],
})

print('Model Comparison: Information Criteria')
print('=' * 80)
print(comparison_ic.to_string(index=False))

# Which model is preferred?
if aic_tvp < aic_fixed:
    print(f'\nAIC prefers: Time-Varying Parameters (delta = {aic_fixed - aic_tvp:.1f})')
else:
    print(f'\nAIC prefers: Fixed Coefficients (delta = {aic_tvp - aic_fixed:.1f})')

if bic_tvp < bic_fixed:
    print(f'BIC prefers: Time-Varying Parameters (delta = {bic_fixed - bic_tvp:.1f})')
else:
    print(f'BIC prefers: Fixed Coefficients (delta = {bic_tvp - bic_fixed:.1f})')

In [ ]:
# Forecast comparison: Fixed vs TVP
n_forecast = 8

# For both models, we need to create future X matrix
future_trend = np.arange(n + 1, n + 1 + n_forecast, dtype=np.float64)
# Repeat quarterly pattern
future_quarters = np.tile([1, 2, 3, 4], n_forecast // 4 + 1)[:n_forecast]
# Adjust to continue from last quarter
last_q = quarters[-1]
future_quarters = np.array([(last_q + i) % 4 + 1 for i in range(1, n_forecast + 1)])

future_X = np.column_stack([
    np.ones(n_forecast),
    future_trend,
    (future_quarters == 2).astype(np.float64),
    (future_quarters == 3).astype(np.float64),
    (future_quarters == 4).astype(np.float64),
])

# Fixed model forecast: y_hat = X_future @ beta_hat
beta_fixed = results_fixed.params[:-1]
sigma2_fixed = results_fixed.params[-1]
fc_fixed = future_X @ beta_fixed
fc_se_fixed = np.sqrt(sigma2_fixed)  # Simple forecast SE

# TVP model forecast: use last smoothed state
beta_last = results_tvp.smoothed_state[-1]  # Last smoothed coefficients
P_last = results_tvp.smoothed_cov[-1]
sigma2_tvp = results_tvp.params[0]

fc_tvp = np.zeros(n_forecast)
fc_var_tvp = np.zeros(n_forecast)

# Build Q matrix for TVP
k = len(regressor_names)
Q_tvp = np.diag(results_tvp.params[1:1+k])

beta_h = beta_last.copy()
P_h = P_last.copy()

for h in range(n_forecast):
    # State prediction: beta_{T+h} = beta_{T+h-1} (random walk)
    if h > 0:
        P_h = P_h + Q_tvp
    X_h = future_X[h:h+1, :]  # (1, k)
    fc_tvp[h] = float(X_h @ beta_h)
    fc_var_tvp[h] = float(X_h @ P_h @ X_h.T + sigma2_tvp)

fc_se_tvp = np.sqrt(fc_var_tvp)

# Create forecast dates
last_period = pd.Period(df['quarter'].iloc[-1], freq='Q')
fc_periods = [last_period + i for i in range(1, n_forecast + 1)]
fc_dates = pd.PeriodIndex(fc_periods).to_timestamp()

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

n_show = 40
ax.plot(date_idx[-n_show:], y[-n_show:], 'k-', linewidth=1, label='Observed')

# Fixed forecast
ax.plot(fc_dates, fc_fixed, 'r-', linewidth=1.5, label='Forecast (Fixed)', marker='o', markersize=4)
ax.fill_between(fc_dates, fc_fixed - 1.96 * fc_se_fixed, fc_fixed + 1.96 * fc_se_fixed,
                alpha=0.15, color='red')

# TVP forecast
ax.plot(fc_dates, fc_tvp, 'b-', linewidth=1.5, label='Forecast (TVP)', marker='s', markersize=4)
ax.fill_between(fc_dates, fc_tvp - 1.96 * fc_se_tvp, fc_tvp + 1.96 * fc_se_tvp,
                alpha=0.15, color='blue')

ax.axvline(date_idx[-1], color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Gas Consumption')
ax.set_title('Forecast Comparison: Fixed vs Time-Varying Coefficients')
ax.legend()
plt.tight_layout()
plt.show()

print('Forecast values:')
for h in range(n_forecast):
    print(f'  h={h+1}: Fixed={fc_fixed[h]:.1f}, TVP={fc_tvp[h]:.1f}')

## Conclusions

1. **SSM = OLS Equivalence**: When the state innovation variance $Q = 0$
   (fixed coefficients), the state-space regression model produces identical
   estimates to classical OLS. This confirms the RegressionSSM implementation.

2. **Time-Varying Coefficients**: The TVP model allows regression coefficients
   to evolve as random walks. The smoothed coefficient paths reveal whether
   the relationship between gas consumption and the regressors changes over time.

3. **Model Selection**: AIC and BIC provide guidance on whether the added
   flexibility of time-varying coefficients is warranted by the data.
   The TVP model uses more effective parameters but may capture structural
   changes that the fixed model misses.

4. **Forecasting**: TVP forecasts adapt to the most recent coefficient values,
   while fixed forecasts project the global average relationship. The TVP
   confidence intervals widen over time as coefficient uncertainty accumulates.

### References
- Harvey, A.C. (1989). *Forecasting, Structural Time Series Models and the Kalman Filter*.
- Durbin, J. & Koopman, S.J. (2012). *Time Series Analysis by State Space Methods*.
- Stock, J.H. & Watson, M.W. (1996). Evidence on Structural Instability in
  Macroeconomic Time Series Relations. *JBES*.